In [6]:
# Read 1000A-2027_de.csv
import re, io, csv, pathlib
import pandas as pd
import numpy as np


_12DIG = re.compile(r"^\d{12}$")

# ---------- robust text read ----------
def _read_text_utf8(path, prefer_sep=";"):
    b = pathlib.Path(path).read_bytes()
    # utf-8-sig eats the BOM if present; fallback to utf-8
    try:
        text = b.decode("utf-8-sig")
    except UnicodeDecodeError:
        text = b.decode("utf-8", errors="ignore")

    # normalize line endings and strip NUL/NBSP
    if "\n" not in text and "\r" in text:
        text = text.replace("\r", "\n")
    text = text.replace("\r\n", "\n")
    text = text.replace("\x00", "").replace("\u00a0", " ")

    # verify sep (you said it's ';' but let's be safe on weird files)
    head = "\n".join(text.splitlines()[:10])
    if head.count(prefer_sep) < 3:
        counts = {s: head.count(s) for s in [";", "\t", ","]}
        prefer_sep = max(counts, key=counts.get)

    return text, prefer_sep

def _read_csv_anycols(text: str, sep: str) -> pd.DataFrame:
    # compute max field count to force stable column width
    lines = text.splitlines()
    # ignore very short metadata lines when computing width, but keep them in the CSV
    max_fields = max((ln.count(sep) for ln in lines[0:2000]), default=0) + 1
    names = list(range(max_fields))
    return pd.read_csv(
        io.StringIO(text),
        sep=sep,
        header=None,
        names=names,            # fixes "Expected 1 fields..." by padding short rows
        dtype=str,
        quoting=csv.QUOTE_NONE, # no special quoting in these exports
        keep_default_na=False,  # keep "-" etc., we'll handle ourselves
        na_filter=False,
        low_memory=False,
    )

# ---------- structure helpers ----------
def _row_has_many_geo_codes(row: pd.Series) -> bool:
    c = 0
    for cell in row.fillna(""):
        parts = str(cell).split()
        if parts and _12DIG.match(parts[0]):
            c += 1
    return c >= 3

def _find_block_bounds(df: pd.DataFrame):
    first_data = df.index[(df[0] == "Insgesamt") & (df[1] == "Insgesamt")]
    if len(first_data) == 0:
        raise ValueError("Couldn't find 'Insgesamt / Insgesamt' start.")
    start_total = int(first_data[0])

    male_hdr = df.index[(df[0] == "Männlich") & (df[1] == "Insgesamt")]
    female_hdr = df.index[(df[0] == "Weiblich") & (df[1] == "Insgesamt")]
    if len(male_hdr) == 0 or len(female_hdr) == 0:
        raise ValueError("Couldn't find Männlich/Weiblich markers.")
    start_male, start_female = int(male_hdr[0]), int(female_hdr[0])

    footer_idx = df.index[df[0].fillna("").astype(str).str.startswith(("©","Stand:","__________"))]
    footer = int(footer_idx[0]) if len(footer_idx) else len(df)

    return {
        "total":  (start_total,   start_male - 1),
        "male":   (start_male + 1, start_female - 1),
        "female": (start_female + 1, footer - 1),
    }, start_total

def _build_geo_schema(df: pd.DataFrame, data_start_row: int):
    """
    Parse the geo header row (with ARS + name) and the sub-header row (with 'Anzahl', '%', 'e').
    Return schema entries ONLY for value columns (Anzahl or %, but we'll filter to Anzahl later),
    and separately remember the position of the flag column so we can ignore it.
    We DO NOT drop any geos based on flags.
    """
    # find the GEO header line above the first data row
    geo_line_idx = None
    for r in range(data_start_row - 1, max(0, data_start_row - 10), -1):
        if _row_has_many_geo_codes(df.loc[r]):
            geo_line_idx = r
            break
    if geo_line_idx is None:
        raise ValueError("Geo header line not found.")

    sub_line_idx = geo_line_idx + 1
    geo_line = df.loc[geo_line_idx].fillna("")
    sub_line = df.loc[sub_line_idx].fillna("")

    schema = []
    c = 2  # data columns start at 2 (0/1 are group + age)
    max_c = df.shape[1]

    while c < max_c:
        geo_cell = str(geo_line.iloc[c]).strip()
        sub = str(sub_line.iloc[c]).strip().lower()

        if not geo_cell:
            c += 1
            continue

        # Determine geo_id (Deutschland has no 12-digit code)
        parts = geo_cell.split()
        geo_id = parts[0] if (parts and _12DIG.match(parts[0])) else "DE"

        # Only consider actual value columns here; ignore pure flag columns.
        if sub.startswith("anzahl") or sub.startswith("%"):
            measure = "share" if sub.startswith("%") else "anzahl"

            # If the next column in the sub-line is 'e', treat it as the flag column and skip it
            flag_idx = c + 1 if (c + 1 < max_c and str(sub_line.iloc[c + 1]).strip().lower() in {"e", ""}) else None

            schema.append({
                "value_col": c,
                "flag_col": flag_idx,   # may be None
                "geo_id": geo_id,
                "measure": measure,
            })

            # advance: skip the neighboring flag column if present
            c = (flag_idx + 1) if flag_idx is not None else (c + 1)
        else:
            # This column isn’t a value (likely the 'e' flag); skip it
            c += 1

    return schema

def _clean_num_zero(s):
    if s is None:
        return 0.0
    s = str(s).strip()
    if s in {"", "-", "–"}:
        return 0.0
    s = s.replace(",", ".")
    try:
        return float(s)
    except Exception:
        return 0.0

def _block_to_matrix(df: pd.DataFrame, start: int, end: int, schema):
    """
    Build a wide matrix for one block (Total/Male/Female):
    - rows: ages
    - cols: geo_ids
    - values: Anzahl (value columns only)
    Flag columns are ignored; we do NOT drop geos because of flags.
    """
    # collect unique ages (in file order)
    ages = []
    age_rows = []
    for r in range(start, end + 1):
        age = str(df.iat[r, 1]).strip()
        if not age or age.lower().startswith("davon"):
            continue
        ages.append(age)
        age_rows.append(r)

    # initialize data arrays for each geo_id found in schema (Anzahl only)
    value_specs = [s for s in schema if s["measure"] == "anzahl"]
    geo_ids = [s["geo_id"] for s in value_specs]
    geo_ids_unique = list(dict.fromkeys(geo_ids))  # preserve order, remove dups

    data = {gid: [0.0] * len(ages) for gid in geo_ids_unique}

    # fill values
    for idx, r in enumerate(age_rows):
        for s in value_specs:
            gid = s["geo_id"]
            c = s["value_col"]
            # robust numeric parse: '-' → 0, comma decimals → dot
            val_raw = df.iat[r, c]
            v = _clean_num_zero(val_raw)
            data[gid][idx] = v

    mat = pd.DataFrame(data, index=ages)
    mat.index.name = "age_label"
    return mat.sort_index(axis=1)

# ---------- public API ----------
def load_age_csv_to_matrices(path: str):
    text, sep = _read_text_utf8(path, prefer_sep=";")
    raw = _read_csv_anycols(text, sep)
    # drop empty rows/cols and trim whitespace
    raw = raw.dropna(how="all", axis=0).dropna(how="all", axis=1).reset_index(drop=True)
    raw = raw.applymap(lambda x: x.strip() if isinstance(x, str) else x)

    bounds, first_data_row = _find_block_bounds(raw)
    schema = _build_geo_schema(raw, first_data_row)

    total_df  = _block_to_matrix(raw, *bounds["total"],  schema)
    male_df   = _block_to_matrix(raw, *bounds["male"],   schema)
    female_df = _block_to_matrix(raw, *bounds["female"], schema)

    # optional: ensure identical row order across all three
    all_ages = total_df.index.union(male_df.index).union(female_df.index)
    total_df  = total_df.reindex(all_ages)
    male_df   = male_df.reindex(all_ages)
    female_df = female_df.reindex(all_ages)

    return total_df, male_df, female_df


total_df, male_df, female_df = load_age_csv_to_matrices(
    r"C:\Users\petre\PycharmProjects\cleancensus\additional_data\gender\1000A-2027_de\1000A-2027_de.csv"
)
total_df  = total_df.drop(columns=["DE"], errors="ignore")
male_df   = male_df.drop(columns=["DE"], errors="ignore")
female_df = female_df.drop(columns=["DE"], errors="ignore")

total_df.to_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_totals.pickle")
male_df.to_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_males.pickle")
female_df.to_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_females.pickle")

print(total_df.shape, male_df.shape, female_df.shape)

# geos present in all three?
common = set(total_df.columns) & set(male_df.columns) & set(female_df.columns)
only_in_total  = set(total_df.columns)  - common
only_in_male   = set(male_df.columns)   - common
only_in_female = set(female_df.columns) - common
print("only_in_total:",  len(only_in_total))
print("only_in_male:",   len(only_in_male))
print("only_in_female:", len(only_in_female))

# if you want strict alignment for the residual check:
all_geos = sorted(set(total_df.columns) | set(male_df.columns) | set(female_df.columns))
all_ages = sorted(set(total_df.index)   | set(male_df.index)   | set(female_df.index))

total  = total_df .reindex(index=all_ages, columns=all_geos, fill_value=0.0)
male   = male_df  .reindex(index=all_ages, columns=all_geos, fill_value=0.0)
female = female_df.reindex(index=all_ages, columns=all_geos, fill_value=0.0)

diff = total - (male + female)
print("max abs residual:", diff.abs().to_numpy().max())
print("example 010010000000 present in female?:", "010010000000" in female.columns)


C:\Users\petre\AppData\Local\Temp\ipykernel_36444\1131383290.py:194: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  raw = raw.applymap(lambda x: x.strip() if isinstance(x, str) else x)


(102, 10786) (102, 10786) (102, 10786)
only_in_total: 0
only_in_male: 0
only_in_female: 0
max abs residual: nan
example 010010000000 present in female?: True


In [7]:
# Cleanup and smoke tests

import re
import pandas as pd
import numpy as np


total = pd.read_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_totals.pickle")
male=pd.read_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_males.pickle")
female=pd.read_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_females.pickle")

# --- 2) Rename age rows to numeric (keep 'Insgesamt')
_age_re = re.compile(r"^\s*(\d+)\s+Jahr")

def _canon_age(lbl: str):
    s = str(lbl).strip()
    sl = s.lower()
    if sl.startswith("unter 1 jahr"):          return "0"
    if sl.startswith("100 jahre und älter"):   return "100"
    if sl == "insgesamt":                      return "Insgesamt"
    m = _age_re.match(s)
    if m:                                      return str(int(m.group(1)))
    raise ValueError(f"Unrecognized age label: {lbl!r}")

def _rename(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.index = [ _canon_age(x) for x in out.index ]
    # order ages 0..100 then 'Insgesamt' (if present)
    ages = [str(i) for i in range(0, 101)]
    order = [a for a in ages if a in out.index]
    if "Insgesamt" in out.index: order.append("Insgesamt")
    return out.reindex(order)

total  = _rename(total)
male   = _rename(male)
female = _rename(female)

total.to_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_totals.pickle")
male.to_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_males.pickle")
female.to_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_females.pickle")

# --- 3) Verify identical labels (you said shapes already match; we assert)
assert total.shape == male.shape == female.shape, "Shapes differ."
assert total.index.equals(male.index) and total.index.equals(female.index), "Row labels differ."
assert total.columns.equals(male.columns) and total.columns.equals(female.columns), "Column labels differ."

# --- 4) Residuals
combined = male + female
diff = total - combined

tol = 1e-6
abs_diff = diff.abs()
n_bad = int((abs_diff > tol).to_numpy().sum())
max_val = abs_diff.to_numpy().max() if n_bad else 0.0
print(f"[CHECK] |total - (male+female)| > {tol}: {n_bad} cells; max abs diff = {max_val:.6f}")

if n_bad:
    # worst single cell
    idxmax = np.unravel_index(abs_diff.values.argmax(), diff.shape)
    worst_age = diff.index[idxmax[0]]
    worst_geo = diff.columns[idxmax[1]]
    print(f"   worst cell: age='{worst_age}', geo='{worst_geo}', diff={diff.iat[idxmax[0], idxmax[1]]:.6f}")

    # per-geo and per-age aggregates (helpful to spot systematic issues)
    per_geo = diff.sum(axis=0).abs().sort_values(ascending=False).head(10)
    per_age = diff.sum(axis=1).abs().sort_values(ascending=False).head(10)
    print("\nTop 10 |sum over ages| per geo:\n", per_geo.to_string())
    print("\nTop 10 |sum over geos| per age:\n", per_age.to_string())
else:
    print("All good: totals equal male+female within tolerance.")

# --- 5) Save full residual matrix
out_csv = r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_total_minus_male_female.csv"
diff.to_csv(out_csv)
print(f"\nSaved full diff matrix to: {out_csv}")


[CHECK] |total - (male+female)| > 1e-06: 826693 cells; max abs diff = nan
   worst cell: age='Insgesamt', geo='010010000000', diff=nan

Top 10 |sum over ages| per geo:
 145220020020    100.0
073405001021     89.0
091870117117     88.0
072325005042     83.0
120675701528     83.0
160630092092     82.0
072315001077     82.0
066330011011     80.0
051620024024     79.0
096770155155     78.0

Top 10 |sum over geos| per age:
 13    585.0
89    559.0
48    533.0
62    512.0
47    452.0
15    430.0
2     420.0
72    411.0
17    407.0
5     403.0

Saved full diff matrix to: C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_total_minus_male_female.csv


In [9]:
# More detailed smoke test analysis

import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender")
DIFF_CSV = BASE / "gemeinde_total_minus_male_female.csv"

# 1) Load residual matrix: rows=ages ('0'..'100','Insgesamt'), cols=geo IDs
diff = pd.read_csv(DIFF_CSV, index_col=0)
# ensure numeric
diff = diff.apply(pd.to_numeric, errors="coerce").fillna(0.0)


# 2) Overall stats
tol = 1e-6
abs_diff = diff.abs()
n_rows, n_cols = diff.shape
n_cells = n_rows * n_cols
n_bad = int((abs_diff > tol).to_numpy().sum())
share_bad = n_bad / n_cells

max_val = abs_diff.to_numpy().max()
min_val = diff.to_numpy().min()
max_pos = diff.to_numpy().max()

# where is worst absolute deviation?
ij = np.unravel_index(abs_diff.values.argmax(), diff.shape)
worst_age = diff.index[ij[0]]
worst_geo = diff.columns[ij[1]]
worst_val = diff.iat[ij[0], ij[1]]

print("=== OVERALL ===")
print(f"shape: {n_rows} ages x {n_cols} geos ({n_cells:,} cells)")
print(f"|diff| > {tol}: {n_bad:,} cells ({share_bad:.2%})")
print(f"max |diff|: {max_val:.3f}  (worst at age='{worst_age}', geo='{worst_geo}', diff={worst_val:.3f})")
print(f"min diff: {min_val:.3f}   max diff: {max_pos:.3f}")

# 3) Per-geo summaries
per_geo_sum      = diff.sum(axis=0)          # signed bias per geo
per_geo_abs_sum  = abs_diff.sum(axis=0)      # magnitude over all ages
per_geo_max_abs  = abs_diff.max(axis=0)      # worst age for each geo
per_geo_nonzero  = (abs_diff > tol).sum(axis=0)

print("\n=== TOP GEO OFFENDERS (by |sum over ages|) ===")
print(per_geo_abs_sum.sort_values(ascending=False).head(15).to_string())

print("\n=== TOP GEO WORST SINGLE-AGE DEVIATION (by max |diff|) ===")
print(per_geo_max_abs.sort_values(ascending=False).head(15).to_string())

# 4) Per-age summaries
per_age_sum      = diff.sum(axis=1)          # signed bias per age across geos
per_age_abs_sum  = abs_diff.sum(axis=1)
per_age_max_abs  = abs_diff.max(axis=1)
per_age_nonzero  = (abs_diff > tol).sum(axis=1)

print("\n=== TOP AGE OFFENDERS (by |sum over geos|) ===")
print(per_age_abs_sum.sort_values(ascending=False).head(15).to_string())

print("\n=== TOP AGES WORST SINGLE-GEO (by max |diff|) ===")
print(per_age_max_abs.sort_values(ascending=False).head(15).to_string())

# 5) Specific checks that often reveal parsing issues
if "Insgesamt" in diff.index:
    inz = diff.loc["Insgesamt"]
    print("\n=== 'Insgesamt' row check ===")
    print(f"  max |diff|: {inz.abs().max():.3f},  #nonzero geos: {(inz.abs() > tol).sum()}/{inz.size}")

# columns/ages that are entirely zero (good = perfect match, really not needed, just for reference)
zero_geos = (abs_diff > tol).sum(axis=0) == 0
zero_ages = (abs_diff > tol).sum(axis=1) == 0
print(f"\nGeos with perfect match (all ages): {int(zero_geos.sum())}/{n_cols}")
print(f"Ages with perfect match (all geos): {int(zero_ages.sum())}/{n_rows}")

# 6) Write helper CSVs for deeper inspection
(per_geo_abs_sum.rename("abs_sum_over_ages")
          .to_csv(BASE / "residuals_per_geo_abs_sum.csv"))
(per_geo_max_abs.rename("max_abs_over_ages")
          .to_csv(BASE / "residuals_per_geo_max_abs.csv"))
(per_age_abs_sum.rename("abs_sum_over_geos")
          .to_csv(BASE / "residuals_per_age_abs_sum.csv"))
(per_age_max_abs.rename("max_abs_over_geos")
          .to_csv(BASE / "residuals_per_age_max_abs.csv"))

print(f"\nSaved details to:\n- {BASE/'residuals_per_geo_abs_sum.csv'}"
      f"\n- {BASE/'residuals_per_geo_max_abs.csv'}"
      f"\n- {BASE/'residuals_per_age_abs_sum.csv'}"
      f"\n- {BASE/'residuals_per_age_max_abs.csv'}")


C:\Users\petre\AppData\Local\Temp\ipykernel_36444\1804157389.py:9: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  diff = pd.read_csv(DIFF_CSV, index_col=0)


=== OVERALL ===
shape: 102 ages x 10786 geos (1,100,172 cells)
|diff| > 1e-06: 826,693 cells (75.14%)
max |diff|: 8.000  (worst at age='2', geo='130765665154', diff=-8.000)
min diff: -8.000   max diff: 8.000

=== TOP GEO OFFENDERS (by |sum over ages|) ===
010550042042    249.0
010550004004    248.0
010530032032    248.0
084355007059    247.0
053700020020    247.0
094740126126    247.0
091800117117    247.0
084265005070    246.0
081175006049    245.0
010020000000    245.0
057620020020    245.0
034510005005    245.0
034590005005    244.0
095720135135    244.0
097715705146    243.0

=== TOP GEO WORST SINGLE-AGE DEVIATION (by max |diff|) ===
010615138035    8.0
073345005006    8.0
130765665154    8.0
071415009023    8.0
010535318048    7.0
100450116116    7.0
010585893052    7.0
091865157130    7.0
010595993092    7.0
094775435151    7.0
083355002026    7.0
032565410017    7.0
095770114114    7.0
096790201201    7.0
091780143143    7.0

=== TOP AGE OFFENDERS (by |sum over geos|) ===
58    

Damit passen Ages und Gender (Gemeinderohdaten) - diffs zwischen total und female+male stammen direkt aus den Zensusdaten und sind extrem vernachlässigbar (einstellig auch in Großstadt).
Damit folgt matching zu den 100m Zellen und application der gender ratios.

In [3]:
# Merge Gemeinde (and Kreis etc) info onto census 100m cells  → Parquet (streaming) instead of pickle as previously, mostly for size

import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import pyarrow as pa
import pyarrow.parquet as pq

# ----------------------------
# Config
# ----------------------------
CELLS_IN = Path(r"C:\Users\petre\PycharmProjects\cleancensus\merged\df100_with_single_years_100_4.pickle")
CELLS_ID_COL = "GITTER_ID_100m"

GEM_GPKG = Path(r"C:\Users\petre\PycharmProjects\cleancensus\additional_data\gender\vg250_12-31.utm32s.gpkg.ebenen\vg250_ebenen_1231\DE_VG250.gpkg")
GEM_LAYER = "v_vg250_gem"

OUT_PARQUET = Path(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\cells_100m_with_gemeinde_100_4.parquet")
CHUNK_SIZE = 1_000_000

EPSG_POINTS = 3035   # cell IDs are LAEA Europe (EPSG:3035)
EPSG_GEM    = 25832  # UTM32N (expected for GeoPackage)

# ----------------------------
# Helpers
# ----------------------------

def normalize_objects(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == "object":
            s = df[col]

            # 1) decode bytes → str
            if s.map(lambda x: isinstance(x, (bytes, bytearray))).any():
                s = s.map(lambda x: x.decode("utf-8", "replace") if isinstance(x, (bytes, bytearray)) else x)

            # 2) try numeric if it looks numeric for most rows
            num = pd.to_numeric(s, errors="coerce")
            if num.notna().sum() >= 0.95 * s.notna().sum():
                df[col] = num
            else:
                # fall back to string (Arrow-friendly)
                df[col] = s.astype("string[pyarrow]")
    return df

def parse_centroids_3035(df: pd.DataFrame, id_col: str) -> gpd.GeoDataFrame:
    """From IDs like 'CRS3035RES100mN2689100E4337000' -> centroid points (+50,+50)."""
    ne = df[id_col].astype(str).str.extract(r"N(\d+)E(\d+)", expand=True)
    if ne.isna().any().any():
        bad = df.loc[ne.isna().any(axis=1), id_col].head(5).tolist()
        raise ValueError(f"Could not parse N/E from: {bad}")
    n = ne[0].astype(np.int64) + 50
    e = ne[1].astype(np.int64) + 50
    return gpd.GeoDataFrame(df[[id_col]].copy(),
                            geometry=gpd.points_from_xy(e, n, crs=f"EPSG:{EPSG_POINTS}"))

def require_exact_12_digit_key(gem: gpd.GeoDataFrame) -> str:
    """Validate Gemeinde key column: exactly 12 numeric digits. No padding/transform."""
    col = "Regionalschlüssel_ARS"  # as in your dataset
    if col not in gem.columns:
        raise KeyError(f"Missing '{col}' on Gemeinde layer.")
    s = gem[col].astype(str)
    ok = s.str.fullmatch(r"\d{12}")
    if not ok.all():
        bad = gem.loc[~ok, [col]].head(5)
        raise ValueError(f"Gemeinde key '{col}' must be exactly 12 digits. Examples:\n{bad}")
    return col

def add_ars_parts(gem: gpd.GeoDataFrame, key_col: str) -> gpd.GeoDataFrame:
    """Derive fixed-length parts from the already-validated 12-digit key."""
    ars = gem[key_col].astype(str)
    gem = gem.copy()
    gem["RegionalSchlüssel_ARS"]        = ars
    gem["Land"]                         = ars.str.slice(0, 2)
    gem["Regierungsbezirk"]             = ars.str.slice(2, 3)
    gem["Kreis"]                        = ars.str.slice(3, 5)
    gem["VerwaltungsgemeinschaftTeil1"] = ars.str.slice(5, 7)
    gem["VerwaltungsgemeinschaftTeil2"] = ars.str.slice(7, 9)
    gem["Gemeinde"]                     = ars.str.slice(9, 12)

    expected = {
        "RegionalSchlüssel_ARS": 12, "Land": 2, "Regierungsbezirk": 1, "Kreis": 2,
        "VerwaltungsgemeinschaftTeil1": 2, "VerwaltungsgemeinschaftTeil2": 2, "Gemeinde": 3
    }
    for c, L in expected.items():
        if not (gem[c].astype(str).str.len() == L).all():
            raise ValueError(f"{c} has wrong length; expected {L}.")
    return gem

# ----------------------------
# Load Gemeinden (EPSG:25832 -> 3035)
# ----------------------------
gem = gpd.read_file(GEM_GPKG, layer=GEM_LAYER)
if gem.crs is None:
    raise ValueError("Gemeinde layer has no CRS. Expected EPSG:25832.")
if gem.crs.to_epsg() != EPSG_GEM:
    raise ValueError(f"Gemeinde layer CRS is {gem.crs}, expected EPSG:{EPSG_GEM} (UTM32N).")

key_col = require_exact_12_digit_key(gem)
gem = gem.to_crs(EPSG_POINTS)
gem = add_ars_parts(gem, key_col)
gem = gem[[
    "RegionalSchlüssel_ARS", "Land", "Regierungsbezirk", "Kreis",
    "VerwaltungsgemeinschaftTeil1", "VerwaltungsgemeinschaftTeil2",
    "Gemeinde", "geometry"
]].copy()
_ = gem.sindex  # build once

# ----------------------------
# Load full cells and stream join → Parquet (append via writer)
# ----------------------------
cells_all = pd.read_pickle(CELLS_IN)
if CELLS_ID_COL not in cells_all.columns:
    raise KeyError(f"Missing column '{CELLS_ID_COL}' in cells pickle.")

# Define output column order: full original cols + new attrs
new_cols = ["RegionalSchlüssel_ARS","Land","Regierungsbezirk","Kreis",
            "VerwaltungsgemeinschaftTeil1","VerwaltungsgemeinschaftTeil2","Gemeinde"]
out_cols = list(cells_all.columns) + new_cols

# define which columns must be STRING (Arrow large_string)
FORCE_STRING_COLS = {
    "GITTER_ID_100m", "GITTER_ID_1km", "GITTER_ID_10km",
    "RegionalSchlüssel_ARS", "Land", "Regierungsbezirk", "Kreis",
    "VerwaltungsgemeinschaftTeil1", "VerwaltungsgemeinschaftTeil2", "Gemeinde",
}
# add all werterläuternde columns dynamically
FORCE_STRING_COLS |= {c for c in cells_all.columns if c.startswith("werterlaeuternde_Zeichen_")}

def coerce_for_arrow(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # force specific cols to Arrow strings (large_string)
    for c in FORCE_STRING_COLS:
        if c in df.columns:
            df[c] = df[c].astype("string[pyarrow]")
    # keep booleans stable
    if "is_orphan" in df.columns:
        df["is_orphan"] = df["is_orphan"].astype("bool")
    return df


writer = None
writer = None
schema = None

for start in tqdm(range(0, len(cells_all), CHUNK_SIZE)):
    chunk_full = cells_all.iloc[start:start+CHUNK_SIZE, :].copy()
    pts = parse_centroids_3035(chunk_full, CELLS_ID_COL)
    joined = gpd.sjoin(pts, gem, how="left", predicate="within")
    joined = joined.drop(columns=["index_left", "index_right"], errors="ignore")
    attrs = pd.DataFrame(joined.drop(columns="geometry"))[[CELLS_ID_COL,
        "RegionalSchlüssel_ARS","Land","Regierungsbezirk","Kreis",
        "VerwaltungsgemeinschaftTeil1","VerwaltungsgemeinschaftTeil2","Gemeinde"]]
    enriched = chunk_full.merge(attrs, on=CELLS_ID_COL, how="left")

    # enforce stable dtypes
    enriched = coerce_for_arrow(enriched)
    # (optional) any remaining object cols that should be numbers:
    # enriched[num_cols] = enriched[num_cols].apply(pd.to_numeric, errors="coerce")

    table = pa.Table.from_pandas(enriched[out_cols], preserve_index=False, schema=schema)
    if writer is None:
        # freeze schema from the *first* chunk
        schema = table.schema
        writer = pq.ParquetWriter(OUT_PARQUET.as_posix(), schema, compression="zstd")
    writer.write_table(table)

writer.close()


print(f"Done. Wrote {OUT_PARQUET}")


C:\Users\petre\.conda\envs\MATSimPipeline\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()
100%|██████████| 4/4 [01:00<00:00, 15.19s/it]

Done. Wrote C:\Users\petre\PycharmProjects\cleancensus\with_gender\cells_100m_with_gemeinde_100_4.parquet


In [ ]:
# Manual check

import pandas as pd
df = pd.read_parquet(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\cells_100m_with_gemeinde_100_4.parquet")
print(">>> df.shape:", df.shape)
print(">>> df.columns:", df.columns)
# 3,148,482 rows

# take 20 random rows without replacement
df_sample = df.sample(n=20, random_state=42)  # drop random_state for non-reproducible
print(df_sample)


>>> df.shape: (3148482, 337)
>>> df.columns: Index(['GITTER_ID_100m',
       'Insgesamt_Bevoelkerung_Alter_in_10er-Jahresgruppen_100m-Gitter',
       'Unter10_Alter_in_10er-Jahresgruppen_100m-Gitter',
       'a10bis19_Alter_in_10er-Jahresgruppen_100m-Gitter',
       'a20bis29_Alter_in_10er-Jahresgruppen_100m-Gitter',
       'a30bis39_Alter_in_10er-Jahresgruppen_100m-Gitter',
       'a40bis49_Alter_in_10er-Jahresgruppen_100m-Gitter',
       'a50bis59_Alter_in_10er-Jahresgruppen_100m-Gitter',
       'a60bis69_Alter_in_10er-Jahresgruppen_100m-Gitter',
       'a70bis79_Alter_in_10er-Jahresgruppen_100m-Gitter',
       ...
       'AGE_98', 'AGE_99', 'AGE_100', 'RegionalSchlüssel_ARS', 'Land',
       'Regierungsbezirk', 'Kreis', 'VerwaltungsgemeinschaftTeil1',
       'VerwaltungsgemeinschaftTeil2', 'Gemeinde'],
      dtype='object', length=337)


In [ ]:
import pandas as pd
df = pd.read_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_totals.pickle")
print(">>> df.shape:", df.shape)
print(">>> df.columns:", df.columns)
print("")

>>> df.shape: (102, 10786)
>>> df.columns: Index(['010010000000', '010020000000', '010030000000', '010040000000',
       '010510011011', '010510044044', '010515163003', '010515163010',
       '010515163012', '010515163016',
       ...
       '160775009047', '160775009049', '160775050012', '160775050017',
       '160775050039', '160775051011', '160775051023', '160775051036',
       '160775052003', '160775052043'],
      dtype='object', length=10786)


In [3]:
# Add gender information to 100m cells

from itertools import islice
import numpy as np
import pandas as pd
from tqdm import tqdm

df100_with_ages = pd.read_parquet(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\cells_100m_with_gemeinde_100_4.parquet")
gemeinde_male_df = pd.read_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_males.pickle")
gemeinde_female_df = pd.read_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_females.pickle")

# -------- inputs --------
cells = df100_with_ages.copy()  # columns: AGE_0..AGE_100, RegionalSchlüssel_ARS, ...
gem_m = gemeinde_male_df.copy()     # shape: (ages, ARS)
gem_f = gemeinde_female_df.copy()   # shape: (ages, ARS)

ARS_COL   = "RegionalSchlüssel_ARS"
AGE_TOP   = 100
AGE_COLS  = [f"AGE_{a}" for a in range(AGE_TOP + 1)]
M_COLS    = [f"M_{c}" for c in AGE_COLS]
F_COLS    = [f"F_{c}" for c in AGE_COLS]

# -------- 1) build per-age male share by ARS (aligned to AGE_0..AGE_100) --------
# unify row index (ages) to int and restrict to 0..100
def _as_age_index(df):
    idx = pd.to_numeric(df.index, errors="coerce").astype("Int64")
    df2 = df.copy()
    df2.index = idx
    return df2.loc[df2.index.isin(range(AGE_TOP+1))].sort_index()

# ============================================================
# Early vector check: cells ARS vs Gemeinde M/F tables (detailed)
# ============================================================
import numpy as np
import pandas as pd

ARS_COL  = "RegionalSchlüssel_ARS"
AGE_TOP  = 100
AGE_COLS = [f"AGE_{a}" for a in range(AGE_TOP + 1)]

def _norm_ars_series(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip()

def _as_age_index(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce row index to integer ages, keep only 0..AGE_TOP, sorted."""
    out = df.copy()
    out.index = pd.to_numeric(out.index, errors="coerce").astype("Int64")
    return out.loc[out.index.isin(range(AGE_TOP + 1))].sort_index()

def early_gemeinde_vector_check(cells: pd.DataFrame,
                                gem_m: pd.DataFrame,
                                gem_f: pd.DataFrame,
                                *, max_show: int = 20) -> None:
    # --- Normalize keys
    cells = cells.copy()
    cells[ARS_COL] = _norm_ars_series(cells[ARS_COL])

    gem_m = gem_m.copy()
    gem_f = gem_f.copy()
    gem_m.columns = gem_m.columns.astype(str).str.strip()
    gem_f.columns = gem_f.columns.astype(str).str.strip()

    # --- Age-index coverage check on Gemeinde tables
    gm = _as_age_index(gem_m)
    gf = _as_age_index(gem_f)

    missing_ages_m = sorted(set(range(AGE_TOP + 1)) - set(gm.index.astype(int).tolist()))
    missing_ages_f = sorted(set(range(AGE_TOP + 1)) - set(gf.index.astype(int).tolist()))
    print("\n" + "="*92)
    print("[AGE INDEX COVERAGE]")
    print(f"male:   rows={len(gem_m)} -> usable ages={len(gm)}; missing ages: {missing_ages_m[:max_show]}")
    print(f"female: rows={len(gem_f)} -> usable ages={len(gf)}; missing ages: {missing_ages_f[:max_show]}")

    # --- ARS key sets
    ars_cells_all   = set(cells[ARS_COL].unique())
    ars_cells_valid = {a for a in ars_cells_all if a and a.lower() != 'nan'}
    ars_m = set(gm.columns)
    ars_f = set(gf.columns)
    ars_both = ars_m & ars_f
    ars_union = ars_m | ars_f

    # --- Basic counts
    print("\n[ARS KEY COVERAGE]")
    print(f"cells:  unique ARS (incl. NaNs) = {len(ars_cells_all)} | non-null = {len(ars_cells_valid)}")
    print(f"gem M:  columns={len(ars_m)} | gem F: columns={len(ars_f)} | both={len(ars_both)} | union={len(ars_union)}")

    # --- Cells with NaN ARS
    nan_ars_count = cells[ARS_COL].isna().sum()
    print(f"cells with NaN {ARS_COL}: {nan_ars_count}")

    # --- Direction 1: cells -> gem (cells referencing missing ARS in Gemeinde tables)
    missing_in_gemeinde = ars_cells_valid - ars_both
    print(f"\n[CELLS → GEM] ARS present in cells but missing in both M & F tables: {len(missing_in_gemeinde)}")
    if missing_in_gemeinde:
        print("  sample:", sorted(list(missing_in_gemeinde))[:max_show])

    # quantify impact by total population in those cells
    if missing_in_gemeinde:
        # sum cell totals by ARS over AGE_* (vectorized)
        cell_age_sum = cells[AGE_COLS].apply(pd.to_numeric, errors="coerce").fillna(0.0).sum(axis=1)
        bad_mask = cells[ARS_COL].isin(missing_in_gemeinde)
        pop_bad = float(cell_age_sum[bad_mask].sum())
        pop_all = float(cell_age_sum.sum())
        share = (pop_bad / pop_all) if pop_all > 0 else 0.0
        print(f"  impact: pop_in_missing_ARS = {pop_bad:,.0f} of {pop_all:,.0f} ({share:.3%})")

    # --- Direction 2: gem -> cells (ARS in Gemeinde tables but unused by cells)
    extra_in_gemeinde = ars_both - ars_cells_valid
    print(f"\n[GEM → CELLS] ARS in both M & F tables but not used by any cell: {len(extra_in_gemeinde)}")
    if extra_in_gemeinde:
        print("  sample:", sorted(list(extra_in_gemeinde))[:max_show])

        # quantify impact by Gemeinde totals (sum over ages of M+F)
        gm_use = gm.loc[:, gm.columns.isin(extra_in_gemeinde)]
        gf_use = gf.loc[:, gf.columns.isin(extra_in_gemeinde)]
        gm_use = gm_use.apply(pd.to_numeric, errors="coerce").fillna(0.0)
        gf_use = gf_use.apply(pd.to_numeric, errors="coerce").fillna(0.0)
        pop_extra = float((gm_use.values + gf_use.values).sum())
        pop_union = float((gm.values + gf.values).sum())
        share2 = (pop_extra / pop_union) if pop_union > 0 else 0.0
        print(f"  impact: gem_pop_in_unused_ARS = {pop_extra:,.0f} of {pop_union:,.0f} ({share2:.3%})")

    # --- M/F table asymmetries (columns only in one)
    only_in_m = ars_m - ars_f
    only_in_f = ars_f - ars_m
    print(f"\n[ASYMMETRY] ARS only in M (not in F): {len(only_in_m)} | only in F (not M): {len(only_in_f)}")
    if only_in_m:
        print("  sample M-only:", sorted(list(only_in_m))[:max_show])
    if only_in_f:
        print("  sample F-only:", sorted(list(only_in_f))[:max_show])

    # --- Optional: ARS key length distribution (helps catch formatting issues)
    lengths = pd.Series([len(x) for x in ars_cells_valid])
    if not lengths.empty:
        desc = lengths.describe()
        print("\n[KEY LENGTHS in cells ARS] count={:.0f} min={} max={} mean={:.2f} uniq_lengths={}".format(
            desc["count"], int(desc["min"]), int(desc["max"]), float(desc["mean"]),
            sorted(lengths.unique().tolist())[:max_show]
        ))

    print("="*92 + "\n")

early_gemeinde_vector_check(df100_with_ages, gemeinde_male_df, gemeinde_female_df)


# Now the actual work

gem_m = _as_age_index(gem_m)
gem_f = _as_age_index(gem_f)

# coerce to float, fill NaNs
gem_m = gem_m.apply(pd.to_numeric, errors="coerce").fillna(0.0)
gem_f = gem_f.apply(pd.to_numeric, errors="coerce").fillna(0.0)

# align columns (ARS set) between M and F
common_ars = gem_m.columns.intersection(gem_f.columns)
gem_m = gem_m[common_ars]
gem_f = gem_f[common_ars]

MF = gem_m.values + gem_f.values                      # (A, G)
with np.errstate(divide="ignore", invalid="ignore"):
    share = np.divide(gem_m.values, np.maximum(MF, 1e-12))  # (A, G)
share = np.clip(share, 0.0, 1.0)

# national fallback share per age (vector length A)
nat_m = gem_m.sum(axis=1).to_numpy(float)
nat_f = gem_f.sum(axis=1).to_numpy(float)
with np.errstate(divide="ignore", invalid="ignore"):
    nat_share = np.divide(nat_m, np.maximum(nat_m + nat_f, 1e-12))
nat_share = np.clip(nat_share, 0.0, 1.0)

# package shares into DataFrame indexed by age, columns=ARS
share_df = pd.DataFrame(share, index=gem_m.index, columns=common_ars)

# --------  split cells by ARS groups (vectorized) --------
# Clean keys & ages
cells[ARS_COL] = cells[ARS_COL].astype(str).str.strip()

# Ensure AGE_* are numeric float32 to speed math & save RAM
for c in AGE_COLS:
    cells[c] = pd.to_numeric(cells[c], errors="coerce").fillna(0.0).astype(np.float32)

# Pre-create column names (no big DataFrames)
M_age_cols = [f"M_AGE_{a}" for a in range(AGE_TOP + 1)]
F_age_cols = [f"F_AGE_{a}" for a in range(AGE_TOP + 1)]

# Optional: convert ARS to category for faster .map
cells[ARS_COL] = cells[ARS_COL].astype("category")

for a in tqdm(range(AGE_TOP + 1), desc="Gender split per age"):
    age_col = f"AGE_{a}"
    m_col   = f"M_AGE_{a}"
    f_col   = f"F_AGE_{a}"

    # ARS -> share for this age (Series: index=ARS, value=share)
    s_map = share_df.loc[a].astype(np.float32)  # share_df is (age x ARS)

    # Map shares to all rows; fill missing ARS with national share for this age
    s_vals = cells[ARS_COL].map(s_map)               # pandas C-level hashmap
    s_vals = s_vals.fillna(np.float32(nat_share[a])) # fallback
    s_arr  = s_vals.to_numpy(np.float32, copy=False)

    # Vectorized split
    age_arr = cells[age_col].to_numpy(np.float32, copy=False)
    m_arr   = age_arr * s_arr
    # Write results directly to new columns
    cells[m_col] = m_arr
    cells[f_col] = age_arr - m_arr

# Cheap reconstruction check
mf = cells.filter(like="M_AGE_").to_numpy(np.float32) + cells.filter(like="F_AGE_").to_numpy(np.float32)
aa = cells[AGE_COLS].to_numpy(np.float32)
print("[cell-age recon] max abs diff:", float(np.abs(mf - aa).max()))

# Totals by gender (fast, column-wise)
cells["M_TOTAL"] = cells[M_age_cols].sum(axis=1).astype(np.float32)
cells["F_TOTAL"] = cells[F_age_cols].sum(axis=1).astype(np.float32)

df100_with_gender = cells

df100_with_gender.to_parquet(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\cells_100m_with_gemeinde_with_gender.parquet")

# =========================
# Sanity checks (gendered)
# =========================

ARS_COL   = "RegionalSchlüssel_ARS"
AGE_TOP   = 100
AGE_COLS  = [f"AGE_{a}" for a in range(AGE_TOP + 1)]
M_COLS    = [f"M_{c}" for c in AGE_COLS]
F_COLS    = [f"F_{c}" for c in AGE_COLS]

df = df100_with_gender  # alias

# 0) Basic hygiene: NaN/inf/negatives
def _count_bad(x: pd.Series) -> tuple[int,int,int]:
    v = pd.to_numeric(x, errors="coerce")
    return int(v.isna().sum()), int(np.isinf(v).sum()), int((v < 0).sum())

def _summarize_bad(cols):
    n_na = n_inf = n_neg = 0
    for c in cols:
        a,b,cnt = _count_bad(df[c])
        n_na  += a; n_inf += b; n_neg += cnt
    return n_na, n_inf, n_neg

na_age, inf_age, neg_age = _summarize_bad(AGE_COLS)
na_m,   inf_m,   neg_m   = _summarize_bad(M_COLS)
na_f,   inf_f,   neg_f   = _summarize_bad(F_COLS)
print(f"[hygiene] AGE NaN/inf/neg: {na_age}/{inf_age}/{neg_age} | "
      f"M NaN/inf/neg: {na_m}/{inf_m}/{neg_m} | F NaN/inf/neg: {na_f}/{inf_f}/{neg_f}")

# 1) Per-cell, per-age reconstruction: M + F == AGE
mf = df.loc[:, M_COLS].to_numpy(float) + df.loc[:, F_COLS].to_numpy(float)
aa = df.loc[:, AGE_COLS].to_numpy(float)
delta = mf - aa
print(f"[cell-age recon] max abs diff = {float(np.abs(delta).max()):.3e}, "
      f"mean abs diff = {float(np.abs(delta).mean()):.3e}")

# 2) Row totals consistency (optional: needs/creates totals)
if "M_TOTAL" not in df.columns:
    df["M_TOTAL"] = df.loc[:, M_COLS].sum(axis=1)
if "F_TOTAL" not in df.columns:
    df["F_TOTAL"] = df.loc[:, F_COLS].sum(axis=1)
row_age_sum = df.loc[:, AGE_COLS].sum(axis=1).to_numpy(float)
row_mf_sum  = (df["M_TOTAL"] + df["F_TOTAL"]).to_numpy(float)
rel_row = np.where(np.maximum(row_age_sum, 1.0)>0,
                   np.abs(row_mf_sum - row_age_sum) / np.maximum(row_age_sum, 1.0), 0.0)
print(f"[row totals] max rel err = {float(rel_row.max()):.3e}, mean = {float(rel_row.mean()):.3e}")

# 3) Optional: check against POP_TOTAL_100m_adj if present
if "POP_TOTAL_100m_adj" in df.columns:
    rel_vs_pop = np.where(np.maximum(df["POP_TOTAL_100m_adj"].to_numpy(float), 1.0)>0,
                          np.abs(row_age_sum - df["POP_TOTAL_100m_adj"].to_numpy(float))
                          / np.maximum(df["POP_TOTAL_100m_adj"].to_numpy(float), 1.0), 0.0)
    print(f"[vs POP_TOTAL_100m_adj] max rel err = {float(rel_vs_pop.max()):.3e}, "
          f"mean = {float(rel_vs_pop.mean()):.3e}")

# 4) Aggregate by age over all cells: sum(M)+sum(F) == sum(AGE)
agg_m = df.loc[:, M_COLS].sum(axis=0).to_numpy(float)
agg_f = df.loc[:, F_COLS].sum(axis=0).to_numpy(float)
agg_a = df.loc[:, AGE_COLS].sum(axis=0).to_numpy(float)
agg_delta = (agg_m + agg_f) - agg_a
print(f"[national-by-age] max abs diff = {float(np.abs(agg_delta).max()):.3e}, "
      f"mean abs diff = {float(np.abs(agg_delta).mean()):.3e}")

# 5) By-ARS reconstruction (M+F vs AGE)
by_ars_m = df.groupby(ARS_COL)[M_COLS].sum(min_count=1)
by_ars_f = df.groupby(ARS_COL)[F_COLS].sum(min_count=1)
by_ars_a = df.groupby(ARS_COL)[AGE_COLS].sum(min_count=1)
by_ars_rel = ( (by_ars_m.values + by_ars_f.values) - by_ars_a.values )
by_ars_den = np.maximum(by_ars_a.values, 1.0)
by_ars_rel = np.abs(by_ars_rel) / by_ars_den
# summarize per ARS (max over ages)
per_ars_max = by_ars_rel.max(axis=1) if by_ars_rel.size else np.array([])
print(f"[ARS recon] ARS count={by_ars_a.shape[0]} | max rel err per ARS = "
      f"{float(per_ars_max.max()) if per_ars_max.size else 0.0:.3e} "
      f"| median = {float(np.median(per_ars_max)) if per_ars_max.size else 0.0:.3e}")

# 6) Spot worst offenders (optional print of top 10 ARS by max rel err)
if per_ars_max.size:
    worst_idx = np.argsort(-per_ars_max)[:10]
    worst_ars = by_ars_a.index.to_numpy()[worst_idx]
    worst_vals = per_ars_max[worst_idx]
    print("[ARS recon] worst 10 (ARS, max_rel_err):",
          list(zip(worst_ars.tolist(), [float(x) for x in worst_vals])))

# 7) Non-negativity check (quick)
min_age = float(np.min(aa)) if aa.size else 0.0
min_m   = float(np.min(df.loc[:, M_COLS].to_numpy(float))) if len(df) else 0.0
min_f   = float(np.min(df.loc[:, F_COLS].to_numpy(float))) if len(df) else 0.0
print(f"[non-negativity] min AGE={min_age:.3e}, min M={min_m:.3e}, min F={min_f:.3e}")

#
row_age = cells[AGE_COLS].sum(axis=1).to_numpy(float)
if "POP_TOTAL_100m_adj" in cells.columns:
    pop = pd.to_numeric(cells["POP_TOTAL_100m_adj"], errors="coerce").fillna(0.0).to_numpy(float)
    bad = (pop == 0) & (row_age > 0)
    print("rows with pop_adj==0 but age_sum>0:", int(bad.sum()))
    if bad.any():
        print(cells.loc[np.flatnonzero(bad)[:10], [ARS_COL, "GITTER_ID_100m"] + AGE_COLS[:5]])


if "POP_TOTAL_100m_adj" in df.columns:
    pop = pd.to_numeric(df["POP_TOTAL_100m_adj"], errors="coerce").fillna(0.0).to_numpy(float)

    bad_poppos_age0 = (pop > 0) & (row_age == 0)  # <-- these drive the 1.0 max rel error
    print("rows with pop_adj>0 but age_sum==0:", int(bad_poppos_age0.sum()))
    # quick peek
    if bad_poppos_age0.any():
        cols_show = ["GITTER_ID_100m", "RegionalSchlüssel_ARS", "POP_TOTAL_100m_adj"]
        print(df.loc[bad_poppos_age0, cols_show].head(10))

    # If you have an is_orphan flag, see if it's just those:
    if "is_orphan" in df.columns:
        print("...of which orphans:", int((bad_poppos_age0 & df["is_orphan"].to_numpy(bool)).sum()))




[AGE INDEX COVERAGE]
male:   rows=102 -> usable ages=101; missing ages: []
female: rows=102 -> usable ages=101; missing ages: []

[ARS KEY COVERAGE]
cells:  unique ARS (incl. NaNs) = 10878 | non-null = 10878
gem M:  columns=10786 | gem F: columns=10786 | both=10786 | union=10786
cells with NaN RegionalSchlüssel_ARS: 0

[CELLS → GEM] ARS present in cells but missing in both M & F tables: 92
  sample: ['010539105105', '010609014014', '031519501501', '031539504504', '031549502502', '031549503503', '031549504504', '031549506506', '031559501501', '031589501501', '031599501501', '032559501501', '032559503503', '032559505505', '032559506506', '033549501501', '033549502502', '064319200200', '064359200200', '066339200200']
  impact: pop_in_missing_ARS = 6,883 of 82,716,786 (0.008%)

[GEM → CELLS] ARS in both M & F tables but not used by any cell: 0

[ASYMMETRY] ARS only in M (not in F): 0 | only in F (not M): 0

[KEY LENGTHS in cells ARS] count=10878 min=4 max=12 mean=12.00 uniq_lengths=[4, 12

Gender split per age:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\petre\AppData\Local\Temp\ipykernel_45660\2133771660.py:207: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cells[m_col] = m_arr
C:\Users\petre\AppData\Local\Temp\ipykernel_45660\2133771660.py:208: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cells[f_col] = age_arr - m_arr
C:\Users\petre\AppData\Local\Temp\ipykernel_45660\2133771660.py:207: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whic

[cell-age recon] max abs diff: 3.814697265625e-06


C:\Users\petre\AppData\Local\Temp\ipykernel_45660\2133771660.py:216: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cells["M_TOTAL"] = cells[M_age_cols].sum(axis=1).astype(np.float32)
C:\Users\petre\AppData\Local\Temp\ipykernel_45660\2133771660.py:217: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cells["F_TOTAL"] = cells[F_age_cols].sum(axis=1).astype(np.float32)


[hygiene] AGE NaN/inf/neg: 0/0/0 | M NaN/inf/neg: 0/0/0 | F NaN/inf/neg: 0/0/0
[cell-age recon] max abs diff = 1.907e-06, mean abs diff = 5.158e-10
[row totals] max rel err = 1.246e-06, mean = 1.525e-07
[vs POP_TOTAL_100m_adj] max rel err = 1.000e+00, mean = 1.156e-05
[national-by-age] max abs diff = 1.250e-01, mean abs diff = 2.707e-02


C:\Users\petre\AppData\Local\Temp\ipykernel_45660\2133771660.py:288: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  by_ars_m = df.groupby(ARS_COL)[M_COLS].sum(min_count=1)
C:\Users\petre\AppData\Local\Temp\ipykernel_45660\2133771660.py:289: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  by_ars_f = df.groupby(ARS_COL)[F_COLS].sum(min_count=1)
C:\Users\petre\AppData\Local\Temp\ipykernel_45660\2133771660.py:290: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future def

[ARS recon] ARS count=10878 | max rel err per ARS = 1.198e-07 | median = 1.130e-07
[ARS recon] worst 10 (ARS, max_rel_err): [('071335011025', 1.1976204916663846e-07), ('073375002071', 1.1920865006231907e-07), ('145245135260', 1.1920825215838704e-07), ('146255231580', 1.1920779741103615e-07), ('034580010010', 1.1920771214590786e-07), ('034540035035', 1.192070016031721e-07), ('082265008101', 1.1920555209599115e-07), ('093730147147', 1.192050405052214e-07), ('065340004004', 1.1920444364932337e-07), ('010625244087', 1.1920413101051963e-07)]
[non-negativity] min AGE=0.000e+00, min M=0.000e+00, min F=0.000e+00
rows with pop_adj==0 but age_sum>0: 0
rows with pop_adj>0 but age_sum==0: 36
                        GITTER_ID_100m RegionalSchlüssel_ARS  \
51785   CRS3035RES100mN2736200E4253000          083355004043   
77405   CRS3035RES100mN2744000E4241000          083355005077   
77406   CRS3035RES100mN2744000E4241100          083355005077   
267507  CRS3035RES100mN2795100E4442800          0918401

The 1.0 rel error is just from the orphaned 100m cells. So we fix those few cells now.

In [4]:
# === Backfill orphan rows (pop>0, age_sum==0) with Gemeinde age shares; recompute M/F ===
import numpy as np
import pandas as pd
from pathlib import Path


INPUT_CELLS_PARQUET = Path(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\cells_100m_with_gemeinde_with_gender_100_4.parquet")
INPUT_GEM_M_PICKLE  = Path(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_males.pickle")
INPUT_GEM_F_PICKLE  = Path(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_females.pickle")

OUTPUT_CELLS_PARQUET = Path(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\cells_100m_with_gender_backfilled_100_4.parquet")
OUTPUT_FILLED_LOG    = Path(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\backfilled_rows_log.csv")

cells = pd.read_parquet(INPUT_CELLS_PARQUET)  # already has AGE_*, M_AGE_*, F_AGE_*, POP_TOTAL_100m_adj, RegionalSchlüssel_ARS, maybe is_orphan
gem_m = pd.read_pickle(INPUT_GEM_M_PICKLE)    # shape: (age 0..100 rows, ARS columns)
gem_f = pd.read_pickle(INPUT_GEM_F_PICKLE)    # shape: (age 0..100 rows, ARS columns)

# --------------------
# Columns / constants
# --------------------
ARS_COL   = "RegionalSchlüssel_ARS"
AGE_COLS  = [f"AGE_{a}" for a in range(101)]
M_COLS    = [f"M_AGE_{a}" for a in range(101)]
F_COLS    = [f"F_AGE_{a}" for a in range(101)]

# ensure keys are clean
cells[ARS_COL] = cells[ARS_COL].astype(str).str.strip()
gem_m.columns = gem_m.columns.astype(str).str.strip()
gem_f.columns = gem_f.columns.astype(str).str.strip()

# --------------------
# Align Gemeinde tables to ages 0..100 and numerics
# --------------------
def _as_age_index(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.index = pd.to_numeric(out.index, errors="coerce").astype("Int64")
    return out.loc[out.index.isin(range(101))].sort_index()

gem_m = _as_age_index(gem_m).apply(pd.to_numeric, errors="coerce").fillna(0.0)
gem_f = _as_age_index(gem_f).apply(pd.to_numeric, errors="coerce").fillna(0.0)

# restrict to common ARS
common_ars = gem_m.columns.intersection(gem_f.columns)
gem_m = gem_m[common_ars]
gem_f = gem_f[common_ars]

# --------------------
# Identify offenders: pop>0 but all ages==0 (usually orphans)
# --------------------
row_age = cells[AGE_COLS].sum(axis=1).to_numpy(float)
pop     = pd.to_numeric(cells["POP_TOTAL_100m_adj"], errors="coerce").fillna(0.0).to_numpy(float)
is_orphan = cells.get("is_orphan", False)
if isinstance(is_orphan, pd.Series):
    is_orphan = is_orphan.to_numpy(bool)
else:
    is_orphan = np.zeros(len(cells), dtype=bool)

off_mask = (pop > 0) & (row_age == 0) & is_orphan
off_idx  = np.flatnonzero(off_mask)
print(f"Backfilling orphan rows: {off_idx.size}")

# save a tiny log of which rows we touch
if off_idx.size:
    cols_show = ["GITTER_ID_100m", ARS_COL, "POP_TOTAL_100m_adj"]
    cells.loc[cells.index[off_idx], cols_show].to_csv(OUTPUT_FILLED_LOG, index=False)

# --------------------
# Build national male share (for later gender fallback) from Gemeinde M/F
# --------------------
MF = (gem_m + gem_f)                                  # (101 x G)
nat_m = gem_m.sum(axis=1).to_numpy(float)             # (101,)
nat_f = gem_f.sum(axis=1).to_numpy(float)
with np.errstate(divide="ignore", invalid="ignore"):
    nat_share = np.divide(nat_m, np.maximum(nat_m + nat_f, 1e-12))  # (101,)
nat_share = np.clip(nat_share, 0.0, 1.0)

# --------------------
# Backfill ages & recompute M/F for offenders only
# --------------------
if off_idx.size:
    # per-ARS age shares from Gemeinde totals (M+F), with national fallback
    G_age_tot = MF.to_numpy(float)            # (101, G)
    nat_age_share = G_age_tot.sum(axis=1)
    nat_age_share = nat_age_share / max(nat_age_share.sum(), 1e-12)  # (101,)

    ars_to_col = {ars: j for j, ars in enumerate(MF.columns)}
    ars_idx = cells.loc[cells.index[off_idx], ARS_COL].map(ars_to_col).to_numpy(object)

    S_age = np.empty((off_idx.size, 101), dtype=np.float64)
    for j, col in enumerate(ars_idx):
        if isinstance(col, (int, np.integer)) and 0 <= col < G_age_tot.shape[1]:
            v = G_age_tot[:, int(col)]
            den = v.sum()
            S_age[j, :] = v/den if den > 0 else nat_age_share
        else:
            S_age[j, :] = nat_age_share

    # Fill AGE_* = pop * S_age
    ages_off = pop[off_idx][:, None] * S_age
    cells.loc[cells.index[off_idx], AGE_COLS] = ages_off

    # Build male-share matrix share_df (age x ARS) from Gemeinde M/F
    with np.errstate(divide="ignore", invalid="ignore"):
        share_df = (gem_m / np.maximum((gem_m + gem_f), 1e-12)).clip(0.0, 1.0)

    # Gather male shares per offender ARS (append national as fallback column)
    share_vals = share_df.to_numpy(np.float32)                               # (101, G)
    share_vals = np.concatenate([share_vals, nat_share.reshape(101,1)], 1)   # (101, G+1)
    fallback_col = share_vals.shape[1] - 1

    ars_to_col_gender = {ars: j for j, ars in enumerate(share_df.columns)}
    col_idx = cells.loc[cells.index[off_idx], ARS_COL].map(ars_to_col_gender).to_numpy(object)
    col_idx = np.array([c if isinstance(c, (int, np.integer)) else fallback_col for c in col_idx], dtype=np.int64)

    S_gender = share_vals[:, col_idx].T.astype(np.float64)                   # (k x 101)
    male_off   = ages_off * S_gender
    female_off = ages_off - male_off

    # Write back M/F ages and totals for offenders
    cells.loc[cells.index[off_idx], M_COLS] = male_off
    cells.loc[cells.index[off_idx], F_COLS] = female_off
    cells.loc[cells.index[off_idx], "M_TOTAL"] = male_off.sum(axis=1)
    cells.loc[cells.index[off_idx], "F_TOTAL"] = female_off.sum(axis=1)

    print(f"Filled {off_idx.size} orphan rows via Gemeinde age shares + male shares.")

# --------------------
# Quick QA (filtered & overall)
# --------------------
row_age = cells[AGE_COLS].sum(axis=1).to_numpy(float)
rel_vs_pop_all = np.abs(row_age - pop) / np.maximum(pop, 1.0)
print(f"[vs POP_TOTAL_100m_adj] max={rel_vs_pop_all.max():.3e} mean={rel_vs_pop_all.mean():.3e}")

ok_mask = ~( (pop > 0) & (row_age == 0) )  # should now be all True
rel_vs_pop_ok = np.abs(row_age[ok_mask] - pop[ok_mask]) / np.maximum(pop[ok_mask], 1.0)
print(f"[vs POP_TOTAL_100m_adj | filtered] max={rel_vs_pop_ok.max():.3e} mean={rel_vs_pop_ok.mean():.3e}")

# sanity: M+F==AGE per age (global)
agg_m = cells[M_COLS].sum(axis=0).to_numpy(float)
agg_f = cells[F_COLS].sum(axis=0).to_numpy(float)
agg_a = cells[AGE_COLS].sum(axis=0).to_numpy(float)
print(f"[national-by-age] max|M+F-AGE|={float(np.abs((agg_m+agg_f)-agg_a).max()):.3e}")

# --------------------
# Save
# --------------------
OUTPUT_CELLS_PARQUET.parent.mkdir(parents=True, exist_ok=True)
cells.to_parquet(OUTPUT_CELLS_PARQUET, index=False)
print("Wrote:", OUTPUT_CELLS_PARQUET)
if off_idx.size:
    print("Backfilled rows log:", OUTPUT_FILLED_LOG)


Backfilling orphan rows: 36


C:\Users\petre\AppData\Local\Temp\ipykernel_45660\83211466.py:100: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.02526988 0.02721774 0.02721774 0.02416747 0.02714932 0.02816788
 0.02835864 0.03660769 0.03941909 0.02739788 0.07665678 0.02790698
 0.03088803 0.04444444 0.02568807 0.0219345  0.02060568 0.02060568
 0.01722076 0.01519337 0.01250521 0.02947598 0.02507837 0.04363089
 0.0310434  0.02439024 0.01558846 0.02854424 0.01552511 0.02496533
 0.         0.02227723 0.02673147 0.         0.         0.        ]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  cells.loc[cells.index[off_idx], AGE_COLS] = ages_off
C:\Users\petre\AppData\Local\Temp\ipykernel_45660\83211466.py:100: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.02613767 0.03689516 0.03689516 0.02842231 0.0268605  0.02

Filled 36 orphan rows via Gemeinde age shares + male shares.
[vs POP_TOTAL_100m_adj] max=3.753e-08 mean=2.645e-09
[vs POP_TOTAL_100m_adj | filtered] max=3.753e-08 mean=2.645e-09
[national-by-age] max|M+F-AGE|=6.361e-05
Wrote: C:\Users\petre\PycharmProjects\cleancensus\with_gender\cells_100m_with_gender_backfilled_100_4.parquet
Backfilled rows log: C:\Users\petre\PycharmProjects\cleancensus\with_gender\backfilled_rows_log.csv


In [7]:
# === Compare input vs output Parquet after backfill === (sanity checks)
import numpy as np
import pandas as pd
from pathlib import Path

IN_PARQUET  = Path(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\cells_100m_with_gemeinde_with_gender_100_4.parquet")
OUT_PARQUET = Path(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\cells_100m_with_gender_backfilled_100_4.parquet")

ID_COL   = "GITTER_ID_100m"
ARS_COL  = "RegionalSchlüssel_ARS"
POP_COL  = "POP_TOTAL_100m_adj"
AGE_TOP  = 100
AGE_COLS = [f"AGE_{a}" for a in range(AGE_TOP+1)]
M_COLS   = [f"M_AGE_{a}" for a in range(AGE_TOP+1)]
F_COLS   = [f"F_AGE_{a}" for a in range(AGE_TOP+1)]

# ---------- helpers ----------
def _ensure_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

def _sum_by_age(df, cols):
    # returns numpy vector length 101 (float64)
    return df[cols].sum(axis=0).to_numpy()

def _row_age_sum(df, cols):
    return df[cols].sum(axis=1).to_numpy()

def _present(df, cols):
    return [c for c in cols if c in df.columns]

def _fmt(x):  # pretty small/large numbers
    return f"{x:,.0f}"

# ---------- load ----------
df_in  = pd.read_parquet(IN_PARQUET)
df_out = pd.read_parquet(OUT_PARQUET)

# sanity on ID column presence
if ID_COL not in df_in.columns or ID_COL not in df_out.columns:
    raise KeyError(f"Missing {ID_COL} in one of the files.")

# align order by ID (safe join on ID only; preserves all rows)
common_cols = [ID_COL, ARS_COL, POP_COL] + list(set(AGE_COLS + M_COLS + F_COLS))
df_in  = df_in.reindex(columns=[c for c in common_cols if c in df_in.columns]).copy()
df_out = df_out.reindex(columns=[c for c in common_cols if c in df_out.columns]).copy()

# numeric coercion
_ensure_numeric(df_in,  [POP_COL] + _present(df_in,  AGE_COLS + M_COLS + F_COLS))
_ensure_numeric(df_out, [POP_COL] + _present(df_out, AGE_COLS + M_COLS + F_COLS))

# ---------- basic shape ----------
print("\n=== SHAPE ===")
print(f"in : rows={len(df_in):,}  cols={df_in.shape[1]}")
print(f"out: rows={len(df_out):,} cols={df_out.shape[1]}")

# ---------- national totals ----------
age_cols_in  = _present(df_in,  AGE_COLS)
age_cols_out = _present(df_out, AGE_COLS)
m_cols_out   = _present(df_out, M_COLS)
f_cols_out   = _present(df_out, F_COLS)

pop_in  = pd.to_numeric(df_in.get(POP_COL, pd.Series(0)), errors="coerce").fillna(0.0).to_numpy(float)
pop_out = pd.to_numeric(df_out.get(POP_COL, pd.Series(0)), errors="coerce").fillna(0.0).to_numpy(float)

nat_age_in  = _sum_by_age(df_in,  age_cols_in)  if age_cols_in  else np.zeros(AGE_TOP+1)
nat_age_out = _sum_by_age(df_out, age_cols_out) if age_cols_out else np.zeros(AGE_TOP+1)

print("\n=== NATIONAL SUMS ===")
print(f"sum(pop) in : {_fmt(pop_in.sum())}")
print(f"sum(pop) out: {_fmt(pop_out.sum())}")
print(f"sum(AGE_*) in : {_fmt(nat_age_in.sum())}")
print(f"sum(AGE_*) out: {_fmt(nat_age_out.sum())}")
print(f"Δ sum(AGE_*) (out-in): {_fmt((nat_age_out - nat_age_in).sum())}")

# top-5 age buckets by absolute change
age_diff = nat_age_out - nat_age_in
top5_idx = np.argsort(-np.abs(age_diff))[:5]
print("top-5 |AGE_a| changes (a, absΔ, signΔ):",
      [(int(i), float(abs(age_diff[i])), float(age_diff[i])) for i in top5_idx])

# ---------- row-level checks ----------
row_age_in  = _row_age_sum(df_in,  age_cols_in)  if age_cols_in  else np.zeros(len(df_in))
row_age_out = _row_age_sum(df_out, age_cols_out) if age_cols_out else np.zeros(len(df_out))

print("\n=== ROW-LEVEL ===")
# orphans fixed: pop>0 & age==0 (in)  vs (out)
pop_pos_in   = pop_in  > 0
pop_pos_out  = pop_out > 0
age_zero_in  = row_age_in  == 0
age_zero_out = row_age_out == 0

orphans_in  = int((pop_pos_in  & age_zero_in ).sum())
orphans_out = int((pop_pos_out & age_zero_out).sum())
print(f"rows with pop>0 & age_sum==0 : in={orphans_in:,}  out={orphans_out:,}")

# rows whose AGE_* changed (tolerance)
eps = 1e-9
changed_rows = int((np.abs(row_age_out - row_age_in) > eps).sum())
print(f"rows with Δ row_age_sum != 0 (>|{eps}|): {changed_rows:,}")

# if IDs match, show sample changed IDs
if len(df_in) == len(df_out):
    # we align by order; optional: align precisely by ID for diff
    in_map  = df_in.set_index(ID_COL)
    out_map = df_out.set_index(ID_COL)
    common_ids = in_map.index.intersection(out_map.index)
    ra_in  = in_map.loc[common_ids, age_cols_in].sum(axis=1).to_numpy() if age_cols_in else np.zeros(len(common_ids))
    ra_out = out_map.loc[common_ids, age_cols_out].sum(axis=1).to_numpy() if age_cols_out else np.zeros(len(common_ids))
    diffs  = np.abs(ra_out - ra_in)
    if diffs.size:
        idx = np.argsort(-diffs)[:10]



=== SHAPE ===
in : rows=3,148,482  cols=306
out: rows=3,148,482 cols=306

=== NATIONAL SUMS ===
sum(pop) in : 82,716,897
sum(pop) out: 82,716,897
sum(AGE_*) in : 82,716,784
sum(AGE_*) out: 82,716,897
Δ sum(AGE_*) (out-in): 111
top-5 |AGE_a| changes (a, absΔ, signΔ): [(54, 2.2245902803260833, 2.2245902803260833), (58, 2.141303959302604, 2.141303959302604), (59, 2.140804024413228, 2.140804024413228), (60, 2.13398264744319, 2.13398264744319), (55, 2.0964994076639414, 2.0964994076639414)]

=== ROW-LEVEL ===
rows with pop>0 & age_sum==0 : in=36  out=0
rows with Δ row_age_sum != 0 (>|1e-09|): 3,086,694


Looks very good. Moving on to all kinds of other binned data. (The backfilled file is much larger only bcs of upcasts. This is fine, we'll downcast at the very end)